# Notebook 2a — EGNN + Distance-GNN training and extraction (Colab, T4)

Runs after Step 0 (`prereg.py`, `seeds.py`, `embed_utils.py` in `src/`, config locked) and Step 1/1b
(`data/splits/split_v1.tsv` frozen). Does **not** touch GNINA — that is notebooks 2b/2c on AWS.

What this notebook does, in order:
1. Mount Drive, load the frozen config and the frozen split.
2. Build one graph per complex (pocket heavy atoms + ligand heavy atoms, cached to disk).
3. Train EGNN and Distance-GNN at the primary seed (`seeds.global`, 20260919).
4. Train the untrained anchors (same seed, zero training) — the floor control.
5. Train the same-architecture anchor-seed models (20260923), **only if** `controls.same_arch_seed_anchor`
   is `true` in the frozen config and `seeds.anchor` is set. If not, this section is skipped and printed
   loudly, because the methods say the anchor-seed deviation must be logged *before* this runs, not after.
6. Validation gate (Pearson r ≥ 0.4, clearly beats untrained) → `logs/table1_performance.tsv`.
7. Rotation + translation invariance smoke test in float64 (diff < 1e-10).
8. Extraction: 64-d embeddings from `readout.0` for train/val/test, every trained/untrained network,
   float64, ordered by complex ID, hashed.
9. Embedding health check (`embed_utils.embedding_sanity`).
10. Commit.

**Assumptions this notebook makes that are not literally spelled out in `config.yaml`** (flagged here
instead of silently baked in — add a `log_deviation_note` for any of these you want on the record):
- The complex graph is pocket heavy atoms (`<pid>_pocket.pdb`, waters excluded) plus ligand heavy atoms
  (`<pid>_ligand.sdf`, falling back to `.mol2`), exactly as the methods table describes for both EGNN and
  Distance-GNN ("same graph"). No hydrogens.
- Radius cutoff for graph construction is 5.0 Å for **both** architectures (the config only names
  `distance_gnn.cutoff_A: 5.0`; the methods table's "same graph" line is read as EGNN sharing that cutoff).
- Node type = element bucket × {protein, ligand}, 8 element buckets (C, N, O, S, P, halogen, metal, other)
  × 2 = 16 types, matching `n_types=16` in the reference models.
- "Clearly beats untrained" (performance gate) = trained validation Pearson r exceeds the untrained
  network's by at least 0.10. This margin is not in the config; it is a documented default — change
  `GATE_MARGIN` below if you have a better-motivated value, and log it if you do.


## 0. Mount Drive, locate project, load config + split

In [ ]:
import os, sys, json, time, hashlib, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Point this at the same project root Step 0/1 used (contains config/, src/, data/, logs/).
PROJECT_DIR = "/content/drive/MyDrive/pdbbind_data"  # matches Step 0 / Step 1b's PROJECT_DIR exactly
ROOT = Path(PROJECT_DIR).resolve()
assert ROOT.exists(), f"Project root not found: {ROOT}. Set PROJECT_DIR to your Step 0 ROOT."
os.chdir(ROOT)
for p in (str(ROOT), str(ROOT / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("root:", ROOT)


In [ ]:
def pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    print(pkgs, "ok" if r.returncode == 0 else "FAILED: " + r.stderr.strip().splitlines()[-1][:200])

pip("torch", "rdkit", "pyyaml", "scipy", "pandas", "tqdm", "scikit-learn")
import numpy as np, pandas as pd, yaml, torch, torch.nn as nn
from scipy.stats import pearsonr
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# These three modules are written by the Step 0 notebook. If this import fails, run Step 0 first --
# this notebook does not recreate them, so a drift between Step 0's src/ and this notebook is impossible.
try:
    from src import prereg
    from src import seeds as seedmod
    from src import embed_utils as eu
except ImportError as e:
    raise ImportError(
        "src/prereg.py, src/seeds.py or src/embed_utils.py not found under ROOT/src. "
        "Run the Step 0 notebook (which writes them) before this one."
    ) from e

cfg = prereg.load_config(ROOT)   # frozen config.yaml if locked, else draft (with a warning printed)
GLOBAL_SEED = int(cfg["seeds"]["global"])
ANCHOR_SEED = cfg["seeds"].get("anchor")   # may be absent
RUN_ANCHOR = bool(cfg["controls"].get("same_arch_seed_anchor")) and ANCHOR_SEED is not None
print("global/primary seed:", GLOBAL_SEED, "| anchor seed:", ANCHOR_SEED, "| run anchor arm:", RUN_ANCHOR)
if bool(cfg["controls"].get("same_arch_seed_anchor")) and ANCHOR_SEED is None:
    print("WARNING: controls.same_arch_seed_anchor is true but seeds.anchor is not set in the config. "
          "The anchor-seed section below will be SKIPPED until that's fixed and the deviation is logged.")


In [ ]:
split = pd.read_csv(ROOT / cfg["split"]["file"], sep="\t", index_col=0)
split.index = split.index.astype(str).str.lower()
AFF = cfg["split"]["affinity_column"]
PDBBIND_DIR = Path(cfg["paths"]["pdbbind_dir"])
print(split["split"].value_counts().to_dict())
print("affinity column:", AFF)

def build_index(base: Path):
    bases = [base] + [d for d in base.iterdir() if d.is_dir() and len(d.name) != 4]
    idx = {}
    for b in bases:
        for d in b.iterdir():
            if d.is_dir() and len(d.name) == 4:
                idx[d.name.lower()] = d
    return idx

IDX = build_index(PDBBIND_DIR)
print("complex dirs found:", len(IDX))

EGNN_HP = cfg["training"]["egnn"]
DGNN_HP = cfg["training"]["distance_gnn"]
CUTOFF_A = float(DGNN_HP.get("cutoff_A", 5.0))  # shared cutoff, see notebook-header note
N_RBF = int(DGNN_HP.get("n_rbf", 16))
print("EGNN hp:", EGNN_HP)
print("Distance-GNN hp:", DGNN_HP)
print("shared radius-graph cutoff (A):", CUTOFF_A)


## 1. Atom typing and graph construction

Node type = element bucket (8) × {protein, ligand} (2) = 16, matching `n_types=16` in the reference
models. Heavy atoms only, waters excluded from the pocket file. One graph per complex, cached to
`data/graph_cache/<pid>.pt` so re-running this notebook does not re-parse PDB/SDF files.

In [ ]:
ELEMENT_BUCKETS = ["C", "N", "O", "S", "P", "HAL", "METAL", "OTHER"]
HALOGENS = {"F", "CL", "BR", "I"}
METALS = {"ZN", "MG", "CA", "MN", "FE", "NA", "K", "CU", "NI", "CO", "CD", "HG", "MO"}

def element_bucket(sym: str) -> int:
    s = sym.strip().upper()
    if s in ("C",): return 0
    if s in ("N",): return 1
    if s in ("O",): return 2
    if s in ("S",): return 3
    if s in ("P",): return 4
    if s in HALOGENS: return 5
    if s in METALS: return 6
    return 7

def node_type(sym: str, is_ligand: bool) -> int:
    return element_bucket(sym) * 2 + (1 if is_ligand else 0)

N_TYPES = len(ELEMENT_BUCKETS) * 2
assert N_TYPES == 16

def parse_pocket_heavy_atoms(pdb_path: Path):
    """Return list of (element_symbol, x, y, z) for heavy, non-water atoms in a PDB file."""
    out = []
    with open(pdb_path) as fh:
        for line in fh:
            if not line.startswith(("ATOM", "HETATM")):
                continue
            resname = line[17:20].strip()
            if resname in ("HOH", "WAT", "H2O"):
                continue
            el = line[76:78].strip() if len(line) >= 78 else ""
            if not el:
                el = line[12:16].strip().lstrip("0123456789")[:1]
            if el.upper() in ("H", "D") or not el:
                continue
            try:
                x = float(line[30:38]); y = float(line[38:46]); z = float(line[46:54])
            except ValueError:
                continue
            out.append((el, x, y, z))
    return out

def load_ligand_mol(path: Path):
    path = str(path)
    if path.endswith(".sdf"):
        m = Chem.SDMolSupplier(path, removeHs=False, sanitize=False)[0]
    else:
        m = Chem.MolFromMol2File(path, removeHs=False, sanitize=False)
    return m

def parse_ligand_heavy_atoms(lig_path: Path):
    mol = load_ligand_mol(lig_path)
    if mol is None or mol.GetNumConformers() == 0:
        raise ValueError(f"ligand unreadable or has no conformer: {lig_path}")
    conf = mol.GetConformer()
    out = []
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() <= 1:
            continue
        p = conf.GetAtomPosition(atom.GetIdx())
        out.append((atom.GetSymbol(), p.x, p.y, p.z))
    return out

def complex_files(pid: str):
    d = IDX.get(pid)
    if d is None:
        return None
    pocket = d / f"{pid}_pocket.pdb"
    lig_sdf = d / f"{pid}_ligand.sdf"
    lig_mol2 = d / f"{pid}_ligand.mol2"
    lig = lig_sdf if lig_sdf.exists() else (lig_mol2 if lig_mol2.exists() else None)
    if not pocket.exists() or lig is None:
        return None
    return pocket, lig

print("element buckets:", ELEMENT_BUCKETS, "| N_TYPES:", N_TYPES)


In [ ]:
GRAPH_CACHE = ROOT / "data" / "graph_cache"
GRAPH_CACHE.mkdir(parents=True, exist_ok=True)

def build_graph(pid: str):
    """Parse one complex into (z: LongTensor[n], x: FloatTensor[n,3], y: float)."""
    files = complex_files(pid)
    if files is None:
        raise FileNotFoundError(f"missing pocket/ligand files for {pid}")
    pocket_path, lig_path = files
    prot_atoms = parse_pocket_heavy_atoms(pocket_path)
    lig_atoms = parse_ligand_heavy_atoms(lig_path)
    if not prot_atoms or not lig_atoms:
        raise ValueError(f"empty atom list for {pid} (protein={len(prot_atoms)}, ligand={len(lig_atoms)})")
    zs, xs = [], []
    for el, x, y, zc in prot_atoms:
        zs.append(node_type(el, is_ligand=False)); xs.append((x, y, zc))
    for el, x, y, zc in lig_atoms:
        zs.append(node_type(el, is_ligand=True)); xs.append((x, y, zc))
    z = torch.tensor(zs, dtype=torch.long)
    x = torch.tensor(xs, dtype=torch.float32)
    y = float(split.loc[pid, AFF])
    return z, x, y

def get_graph(pid: str):
    cache_path = GRAPH_CACHE / f"{pid}.pt"
    if cache_path.exists():
        d = torch.load(cache_path)
        return d["z"], d["x"], d["y"]
    z, x, y = build_graph(pid)
    torch.save({"z": z, "x": x, "y": y}, cache_path)
    return z, x, y


### Precompute the graph cache for every complex

Runs once; subsequent runs just load from `data/graph_cache/`. Failures are logged to
`logs/exclusions.tsv` at stage `graph_build` and dropped from that split going forward — they are a
new exclusion category alongside the ones from Step 1, so record them the same way.

In [ ]:
from tqdm.auto import tqdm

def precompute_split_cache(split_name: str):
    ids = split.index[split.split == split_name].tolist()
    ok, bad = [], []
    for pid in tqdm(ids, desc=f"caching {split_name}"):
        cache_path = GRAPH_CACHE / f"{pid}.pt"
        if cache_path.exists():
            ok.append(pid); continue
        try:
            get_graph(pid)
            ok.append(pid)
        except Exception as e:
            bad.append((pid, f"graph_build failed: {str(e)[:200]}"))
    if bad:
        prereg.log_exclusions(bad, stage="graph_build", root=ROOT)
    print(f"{split_name}: cached {len(ok)}, failed {len(bad)}")
    return ok

CACHED_IDS = {s: precompute_split_cache(s) for s in ("train", "val", "test")}


## 2. Dataset, batching, models

Models are the Step 0 reference `EGNNAffinity` / `DistanceGNN` from `src/embed_utils.py` — swap in your own `nn.Module`s here if you have them; everything downstream (extraction, invariance test, hooks) only needs a `readout.0` `nn.Linear` and the `(z, x, edge_index, batch, n_graphs)` forward signature.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from src.embed_utils import EGNNAffinity, DistanceGNN, PreHeadCapture, radius_edges, embedding_sanity, random_rotation

class ComplexGraphDataset(Dataset):
    def __init__(self, ids):
        self.ids = list(ids)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        pid = self.ids[i]
        z, x, y = get_graph(pid)
        return pid, z, x, y

def make_collate(cutoff):
    def collate(batch):
        pids, zs, xs, ys, bidx = [], [], [], [], []
        for gi, (pid, z, x, y) in enumerate(batch):
            pids.append(pid); zs.append(z); xs.append(x); ys.append(y)
            bidx.append(torch.full((z.shape[0],), gi, dtype=torch.long))
        z = torch.cat(zs); x = torch.cat(xs); b = torch.cat(bidx)
        y = torch.tensor(ys, dtype=torch.float32)
        ei = radius_edges(x, b, cutoff=cutoff)
        return pids, z, x, ei, b, len(batch), y
    return collate

MODEL_SPECS = {
    "egnn": dict(cls=EGNNAffinity, hp=EGNN_HP, kwargs=dict(n_types=N_TYPES, hidden=EGNN_HP["hidden"], n_layers=EGNN_HP["n_layers"])),
    "distance_gnn": dict(cls=DistanceGNN, hp=DGNN_HP,
                         kwargs=dict(n_types=N_TYPES, hidden=DGNN_HP["hidden"], n_layers=DGNN_HP["n_layers"],
                                     n_rbf=N_RBF, cutoff=CUTOFF_A)),
}
print({k: v["kwargs"] for k, v in MODEL_SPECS.items()})


## 3. Train / evaluate loop, with checkpoint + resume

Checkpoints every epoch. A GPU disconnect or an accidental interrupt loses at most one epoch, not the run — re-running the training cell resumes from the last checkpoint automatically.

In [ ]:
CKPT_DIR = ROOT / "data" / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
GATE_MARGIN = 0.10  # see the notebook-header note on this assumption

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    total_loss, n = 0.0, 0
    for pids, z, x, ei, b, ng, y in loader:
        z, x, ei, b, y = z.to(DEVICE), x.to(DEVICE), ei.to(DEVICE), b.to(DEVICE), y.to(DEVICE)
        out = model(z, x, ei, b, ng)
        loss = nn.functional.mse_loss(out, y, reduction="sum")
        total_loss += loss.item(); n += ng
        preds.append(out.cpu().numpy()); trues.append(y.cpu().numpy())
    preds = np.concatenate(preds); trues = np.concatenate(trues)
    r = float(pearsonr(preds, trues)[0]) if len(preds) > 1 else float("nan")
    rmse = float(np.sqrt(np.mean((preds - trues) ** 2)))
    return dict(val_loss=total_loss / max(n, 1), pearson_r=r, rmse=rmse, n=n)

def train_one(name: str, seed: int, untrained: bool = False, tag: str = ""):
    """Trains (or, if untrained=True, just initializes) one model. Returns (model, history dict)."""
    spec = MODEL_SPECS[name]
    hp = spec[

## 4. Run the training arms

Primary seed first, then the untrained anchors, then the same-architecture anchor-seed run — the last of which is skipped unless the config and deviation log already agree it should happen.

In [ ]:
MODELS = {}   # (name, tag) -> trained torch.nn.Module

for name in ("egnn", "distance_gnn"):
    print(f"=== training {name}, primary seed {GLOBAL_SEED} ===")
    model, hist = train_one(name, GLOBAL_SEED, untrained=False, tag="primary")
    MODELS[(name, "primary")] = model


In [ ]:
for name in ("egnn", "distance_gnn"):
    print(f"=== untrained anchor for {name}, seed {GLOBAL_SEED} ===")
    model, _ = train_one(name, GLOBAL_SEED, untrained=True, tag="untrained")
    MODELS[(name, "untrained")] = model


In [ ]:
if RUN_ANCHOR:
    for name in ("egnn", "distance_gnn"):
        print(f"=== training {name}, anchor seed {ANCHOR_SEED} ===")
        model, hist = train_one(name, int(ANCHOR_SEED), untrained=False, tag="anchor")
        MODELS[(name, "anchor")] = model
else:
    print("Skipping the anchor-seed arm: controls.same_arch_seed_anchor / seeds.anchor not both set "
          "in the frozen config. Log the deviation and re-run this cell once that's fixed -- "
          "the same-architecture ceiling will otherwise be missing for these two architectures.")


In [ ]:
for n in ["egnn_primary", "distance_gnn_primary", "egnn_anchor"]:
    s = torch.load(CKPT_DIR / f"{n}.pt", map_location="cpu")
    h = s.get("history", [])
    b = min(h, key=lambda r: r["val_loss"]) if h else None
    print(n, "| epoch", s["epoch"], "| is_best", s.get("is_best", "MISSING"),
          "| best_val_loss", s["best_val_loss"], "| patience", s["patience_ctr"])
    print("   last:", h[-1] if h else None)
    print("   best-epoch row:", b)

## 5. Validation gate and Table 1

In [ ]:
TABLE1_PATH = ROOT / "logs" / "table1_performance.tsv"
val_loader_eval = {}
rows = []
for name in ("egnn", "distance_gnn"):
    val_ids = CACHED_IDS["val"]
    loader = DataLoader(ComplexGraphDataset(val_ids), batch_size=MODEL_SPECS[name]["hp"]["batch_size"],
                        shuffle=False, collate_fn=make_collate(CUTOFF_A))
    metrics = {}
    for tag in ("primary", "untrained", "anchor"):
        if (name, tag) not in MODELS:
            continue
        m = evaluate(MODELS[(name, tag)], loader)
        metrics[tag] = m
        rows.append(dict(model=name, seed_tag=tag, split="val", **m))
        print(f"{name:14s} {tag:10s} val_loss={m['val_loss']:.4f} r={m['pearson_r']:.3f} rmse={m['rmse']:.3f} n={m['n']}")
    if "primary" in metrics:
        r_trained = metrics["primary"]["pearson_r"]
        r_untrained = metrics.get("untrained", {}).get("pearson_r", float("nan"))
        gate = (r_trained >= 0.4) and (r_trained - r_untrained >= GATE_MARGIN)
        print(f"  GATE ({name}, primary): r={r_trained:.3f} >= 0.4 and beats untrained "
              f"({r_untrained:.3f}) by >= {GATE_MARGIN}: {'PASS' if gate else 'FAIL'}")
        rows.append(dict(model=name, seed_tag="gate_primary", split="val",
                         val_loss=None, pearson_r=r_trained, rmse=None, n=None, gate_pass=gate))

table1 = pd.DataFrame(rows)
write_header = not TABLE1_PATH.exists()
table1.to_csv(TABLE1_PATH, sep="\t", mode="a", header=write_header, index=False)
print("appended to", TABLE1_PATH.relative_to(ROOT))


## 6. Invariance smoke test (property test, not data-dependent)

The plan requires EGNN and Distance-GNN embeddings to be unchanged under a random rotation +
translation of coordinates, checked at float64 precision. This is a structural property of the
trained network, so it's run on the primary-seed models with a synthetic batch — real complexes
would work identically, since the property must hold for any input.

In [ ]:
def invariance_check(name, model, seed=1, n_graphs=4, sizes=(30, 45, 62, 51)):
    model64 = MODEL_SPECS[name]["cls"](**MODEL_SPECS[name]["kwargs"]).double()
    model64.load_state_dict({k: v.double() for k, v in model.state_dict().items()})
    model64.eval()
    g = torch.Generator().manual_seed(seed)
    zs, xs, bs = [], [], []
    for i, n in enumerate(sizes):
        zs.append(torch.randint(0, N_TYPES, (n,), generator=g))
        xs.append(torch.randn(n, 3, generator=g, dtype=torch.float64) * 4.0)
        bs.append(torch.full((n,), i, dtype=torch.long))
    z, x, b = torch.cat(zs), torch.cat(xs), torch.cat(bs)
    ei = radius_edges(x, b, cutoff=CUTOFF_A)
    with PreHeadCapture(model64, "readout.0") as cap:
        model64(z, x, ei, b, n_graphs)
        e1 = cap.value.clone()
    R, t = random_rotation(seed + 1, dtype=torch.float64), torch.tensor([1.0, -2.0, 3.0], dtype=torch.float64)
    x2 = x @ R.T + t
    ei2 = radius_edges(x2, b, cutoff=CUTOFF_A)
    with PreHeadCapture(model64, "readout.0") as cap:
        model64(z, x2, ei2, b, n_graphs)
        e2 = cap.value.clone()
    max_diff = (e1 - e2).abs().max().item()
    return max_diff

for name in ("egnn", "distance_gnn"):
    if (name, "primary") not in MODELS:
        continue
    diff = invariance_check(name, MODELS[(name, "primary")])
    status = "PASS" if diff < 1e-10 else "FAIL"
    print(f"[{name}] rotation+translation invariance, float64: max|diff| = {diff:.2e}  -> {status}")
    if status == "FAIL":
        print(f"  WARNING: {name} embeddings depend on pose beyond float64 noise. "
              f"Do not treat this architecture's CKA as pose-independent; note it as a limitation.")


## 7. Extraction

For every trained/untrained network, extract the 64-d `readout.0` input for train, val and test, in
float64, batch size 1, no shuffle — matching `embeddings.extraction` in the config. Saved as `.npy`
with ordered complex IDs and a SHA-256 recorded in `logs/embeddings_manifest.json`.

In [ ]:
EMB_DIR = ROOT / "data" / "embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = ROOT / "logs" / "embeddings_manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}

@torch.no_grad()
def extract_split(name, model, split_name):
    model64 = MODEL_SPECS[name]["cls"](**MODEL_SPECS[name]["kwargs"]).double()
    model64.load_state_dict({k: v.double() for k, v in model.state_dict().items()})
    model64.eval()
    ids = CACHED_IDS[split_name]
    loader = DataLoader(ComplexGraphDataset(ids), batch_size=1, shuffle=False,
                        collate_fn=make_collate(CUTOFF_A))
    rows, ordered_ids = [], []
    for pids, z, x, ei, b, ng, y in loader:
        z, x, ei, b = z.double(), x.double(), ei, b
        with PreHeadCapture(model64, "readout.0") as cap:
            model64(z, x, ei, b, ng)
            rows.append(cap.value.cpu().numpy()[0])
        ordered_ids.append(pids[0])
    return np.stack(rows).astype(np.float64), ordered_ids

def sha256_array(arr: np.ndarray) -> str:
    return hashlib.sha256(arr.tobytes()).hexdigest()

for (name, tag), model in MODELS.items():
    for split_name in ("train", "val", "test"):
        E, ids_order = extract_split(name, model, split_name)
        out_npy = EMB_DIR / f"{name}_{tag}_{split_name}.npy"
        out_ids = EMB_DIR / f"{name}_{tag}_{split_name}_ids.txt"
        np.save(out_npy, E)
        out_ids.write_text("\n".join(ids_order))
        key = f"{name}/{tag}/{split_name}"
        manifest[key] = dict(shape=list(E.shape), sha256=sha256_array(E),
                             npy=str(out_npy.relative_to(ROOT)), ids=str(out_ids.relative_to(ROOT)))
        print(f"{key:28s} shape={E.shape} sha256={manifest[key]['sha256'][:12]}")
        assert E.shape[0] == len(ids_order) == len(CACHED_IDS[split_name]), "row-count mismatch"

manifest_path.write_text(json.dumps(manifest, indent=2))
print("wrote", manifest_path.relative_to(ROOT))


## 8. Embedding health check

In [ ]:
cov_cols = ["complex_atom_count", "ligand_heavy_atom_count"]
health = {}
for (name, tag) in MODELS:
    key = f"{name}/{tag}/test"
    E = np.load(EMB_DIR / f"{name}_{tag}_test.npy")
    ids_order = (EMB_DIR / f"{name}_{tag}_test_ids.txt").read_text().splitlines()
    Z = split.loc[ids_order, cov_cols].to_numpy()
    h = embedding_sanity(E, Z)
    health[key] = h
    print(key, json.dumps(h, indent=2))

(ROOT / "logs" / "embedding_health_2a.json").write_text(json.dumps(health, indent=2))


## 9. Commit

In [ ]:
def sh(*args):
    r = subprocess.run(args, cwd=ROOT, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr).strip()

sh("git", "config", "user.email", "notebook2a@local")
sh("git", "config", "user.name", "notebook2a")
if IN_COLAB:
    sh("git", "config", "--global", "--add", "safe.directory", str(ROOT))
sh("git", "add", "logs", "data/splits")  # embeddings/checkpoints/graph_cache are large -- keep them out of git, rely on Drive
print(sh("git", "commit", "-m", "notebook 2a: EGNN + Distance-GNN trained, gated, extracted")[1][:400])
print("\nReminder: confirm in the Drive web UI that data/checkpoints/, data/embeddings/ and "
      "logs/embeddings_manifest.json exist before closing this session.")
